In [ ]:
import rasterio
from rasterio.enums import Resampling
import numpy as np

## Downsampling RBG/NIR ortho rasters

In [45]:
# Input and output file paths
input_path = "/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/Orthomosaics/orthos_1m/chandalar_240710_nir_ortho_1m.tif"
output_path = "/Users/camryn/Documents/UNC/_Tier1_sites/expanded_Yukon_Flats/Orthomosaics/orthos_resampled_3m/chandalar_240710_nir_ortho_3m.tif"

In [ ]:
# open the input raster
with rasterio.open(input_path) as src:
    # Calculate new transform and shape (eg 3 = 3x coarser)
    scale = 3
    new_transform = src.transform * src.transform.scale(
        (src.width / (src.width / scale)),
        (src.height / (src.height / scale))
    )
    new_width = int(src.width / scale)
    new_height = int(src.height / scale)

    # output metadata
    out_meta = src.meta.copy()
    out_meta.update({
        "height": new_height,
        "width": new_width,
        "transform": new_transform
    })

    # create the output raster
    with rasterio.open(output_path, "w", **out_meta) as dst:
        for i in range(1, src.count + 1):  # Loop through all bands (RGB)
            resampled = src.read(
                i,
                out_shape=(new_height, new_width),
                resampling=Resampling.average  # good for downsampling RGB/continuous data
                # could also use Resampling.nearest (fast), Resampling.bilinear (smooth), Resampling.cubic (sharp)
            )
            dst.write(resampled, i)
